# Visual learning across 80,000 neurons: released-example walkthrough

This notebook has one bounded purpose: inspect the released
`VR2_2021_03_20_1` example traces with the paper's signed, whole-session
per-neuron d′ calculation. The complete Neuromatch teaching notebook is preserved
separately as [`neuromatch_visual_learning_80k_neurons.ipynb`](https://github.com/NeuromatchAcademy/course-content/blob/main/projects/neurons/visual_learning_80k_neurons.ipynb); it
is not rewritten here.

Primary sources: [Nature Figure 1](https://www.nature.com/articles/s41586-025-09180-y#Fig1),
[selectivity analysis in Methods](https://www.nature.com/articles/s41586-025-09180-y#Sec20),
[data availability](https://www.nature.com/articles/s41586-025-09180-y#data-availability), and
[Figshare v2](https://doi.org/10.25378/janelia.28811129.v2).


## Complete published figure

**Zhong et al. (2025), Nature Figure 1.**
[Open the anchored figure and caption](https://www.nature.com/articles/s41586-025-09180-y#Fig1).

![Complete Nature Figure 1](https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41586-025-09180-y/MediaObjects/41586_2025_9180_Fig1_HTML.png)


## Connect to the released files

Add **Zhong et al. 2025 - Neuromatch Team Workspace** as a shortcut in
**My Drive**, then run the next cell. `sql.setup()` reads the pinned 297-file catalog into Pandas and DuckDB; the filesystem layer
catalog. The later load cell requests only the three files named below.


In [ ]:
# @title Connect to the shared Drive and build the relational catalog
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    from google.colab import drive as google_drive
except ImportError:
    pass
else:
    google_drive.mount("/content/drive", force_remount=False)
CODE = os.environ.get(
    "ZHONGDB_CODE",
    "/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace/code",
)
if CODE not in sys.path:
    sys.path.insert(0, CODE)

for name in ("database", "drive", "sql"):
    sys.modules.pop(name, None)
import drive

db = drive.setup()
db.schema()

## Exact released inputs

| Use | Released filename | Figshare file ID | Bytes | Published MD5 | Direct file |
|---|---|---:|---:|---|---|
| 1,000-neuron deconvolved example | `VR2_2021_03_20_1_example_raw_spk.npy` | 54866153 | 97,192,128 | `7e341d96305e3a235213b419f71c576d` | [file](https://ndownloader.figshare.com/files/54866153) |
| supervised Train 1 before-learning behaviour | `Beh_sup_train1_before_learning.npy` | 54183863 | 124,559,852 | `75169b8c4c02f5ed9af3fd492e93b9bd` | [file](https://ndownloader.figshare.com/files/54183863) |
| matching retinotopy | `VR2_2021_03_20_trans.npz` | 54184211 | 2,934,270 | `f8fbb33ee2c9461011306c5072d0b06e` | [file](https://ndownloader.figshare.com/files/54184211) |

The next cell checks every value against the pinned
[Figshare v2 release](https://doi.org/10.25378/janelia.28811129.v2) before any
array is loaded.


In [ ]:
# @title Verify exact filenames, IDs, byte counts, and checksums
RECORDING_ID = "VR2_2021_03_20_1"
EXPERIMENT = "sup_train1_before_learning"
RAW_FILE = "VR2_2021_03_20_1_example_raw_spk.npy"

EXPECTED_FILES = {
    RAW_FILE: (54866153, 97192128, "7e341d96305e3a235213b419f71c576d"),
    "Beh_sup_train1_before_learning.npy": (
        54183863,
        124559852,
        "75169b8c4c02f5ed9af3fd492e93b9bd",
    ),
    "VR2_2021_03_20_trans.npz": (
        54184211,
        2934270,
        "f8fbb33ee2c9461011306c5072d0b06e",
    ),
}

catalog = db.table("files").set_index("filename")
verified_files = {}
for filename, expected in EXPECTED_FILES.items():
    if filename not in catalog.index:
        raise ValueError(f"Catalog has no row for {filename}")
    row = catalog.loc[filename]
    actual = (int(row["figshare_file_id"]), int(row["size_bytes"]), row["md5"])
    if actual != expected:
        raise ValueError(f"Pinned metadata changed for {filename}: {actual!r}")
    verified_files[filename] = row.to_dict()
    print({
        "name": filename,
        "id": actual[0],
        "size_bytes": actual[1],
        "md5": actual[2],
        "direct_url": f"https://ndownloader.figshare.com/files/{actual[0]}",
    })

## Paper estimator, unchanged

The authors' implementation defines, for each neuron,

`d′ = 2 × (mean(leaf1) − mean(circle1)) / (std(leaf1) + std(circle1))`.

It uses every frame in the recording for which `ft_move > 0` and
`ft_CorrSpc` is true, maps stimulus ID 2 to `leaf1` and ID 0 to `circle1`, and
classifies signed selectivity at ±0.3 for the Figure 1 density map. See the
authors' exact [d′ function and valid-frame selection](https://github.com/MouseLand/zhong-et-al-2025/blob/paper/utils.py#L370-L443),
[0.3 threshold and density map](https://github.com/MouseLand/zhong-et-al-2025/blob/paper/utils.py#L394-L416),
and [Methods: selectivity analysis](https://www.nature.com/articles/s41586-025-09180-y#Sec20).

The paper's result uses the full deconvolved population for each recording. This
notebook applies the same estimator to the released 1,000-neuron raw example
from one recording. Its output is therefore an inspection of that
released example, not a reproduction of the paper's cohort estimate.


In [ ]:
# @title Exact whole-session signed d-prime and paper-coordinate transform
DP_THRESHOLD = 0.3


def load_released_example():
    return {
        "activity": db.load_file(RAW_FILE, max_gib=0.2),
        "behavior": db.load(RECORDING_ID, "behavior", experiment=EXPERIMENT),
        "retinotopy": db.load(RECORDING_ID, "retinotopy"),
    }


def paper_dprime_for_released_example(activity, behavior, retinotopy):
    activity = np.asarray(activity, dtype=float)
    if activity.ndim != 2:
        raise ValueError("Released activity must have shape neurons × frames")

    frame_wall = np.asarray(behavior["ft_WallID"])
    frame_move = np.asarray(behavior["ft_move"]) > 0
    frame_corridor = np.asarray(behavior["ft_CorrSpc"], dtype=bool)
    n_frames = min(activity.shape[1], len(frame_wall), len(frame_move), len(frame_corridor))
    activity = activity[:, :n_frames]
    valid = frame_move[:n_frames] & frame_corridor[:n_frames]

    stimulus_id = np.asarray(behavior["stim_id"])
    unique_walls = np.asarray(behavior["UniqWalls"])
    leaf_matches = unique_walls[stimulus_id == 2]
    circle_matches = unique_walls[stimulus_id == 0]
    if len(leaf_matches) == 0 or len(circle_matches) == 0:
        raise ValueError("Released behavior is missing leaf1 or circle1")

    leaf_mask = (frame_wall[:n_frames] == leaf_matches[0]) & valid
    circle_mask = (frame_wall[:n_frames] == circle_matches[0]) & valid
    if min(int(leaf_mask.sum()), int(circle_mask.sum())) < 2:
        raise ValueError("Fewer than two valid frames remain for one stimulus")

    leaf = activity[:, leaf_mask]
    circle = activity[:, circle_mask]
    mean_leaf = np.nanmean(leaf, axis=1)
    mean_circle = np.nanmean(circle, axis=1)
    spread = np.nanstd(leaf, axis=1) + np.nanstd(circle, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        dprime = 2.0 * (mean_leaf - mean_circle) / spread

    xy_t = np.asarray(retinotopy["xy_t"], dtype=float)[: activity.shape[0]]
    iarea = np.asarray(retinotopy["iarea"])[: activity.shape[0]]
    if len(xy_t) != activity.shape[0] or len(iarea) != activity.shape[0]:
        raise ValueError("Retinotopy does not cover every released example neuron")
    cortical_x = -xy_t[:, 1]
    cortical_y = xy_t[:, 0]
    mapped = (iarea != -1) & (iarea != 7)
    leaf_selective = mapped & np.isfinite(dprime) & (dprime >= DP_THRESHOLD)
    circle_selective = mapped & np.isfinite(dprime) & (dprime <= -DP_THRESHOLD)
    return {
        "dprime": dprime,
        "mean_leaf": mean_leaf,
        "mean_circle": mean_circle,
        "cortical_x": cortical_x,
        "cortical_y": cortical_y,
        "iarea": iarea,
        "mapped": mapped,
        "leaf_selective": leaf_selective,
        "circle_selective": circle_selective,
        "leaf_frames": int(leaf_mask.sum()),
        "circle_frames": int(circle_mask.sum()),
    }


In [ ]:
# @title Load the three verified files and compute the released-example result
released_example = load_released_example()
example_result = paper_dprime_for_released_example(**released_example)
print({
    "recording_id": RECORDING_ID,
    "experiment": EXPERIMENT,
    "neurons": int(len(example_result["dprime"])),
    "leaf1_valid_frames": example_result["leaf_frames"],
    "circle1_valid_frames": example_result["circle_frames"],
    "mapped_neurons": int(example_result["mapped"].sum()),
    "leaf1_selective_at_0.3": int(example_result["leaf_selective"].sum()),
    "circle1_selective_at_-0.3": int(example_result["circle_selective"].sum()),
})


In [ ]:
# @title Plot the released-example frame counts, signed d-prime, and locations
def plot_released_example(result):
    leaf_color = "#2E8B57"
    circle_color = "#7A5195"
    neutral = "#9AA2AD"
    figure, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

    bars = axes[0, 0].bar(
        ["leaf1", "circle1"],
        [result["leaf_frames"], result["circle_frames"]],
        color=[leaf_color, circle_color],
    )
    axes[0, 0].bar_label(bars)
    axes[0, 0].set(title="Valid whole-session frames", ylabel="frames")

    finite = result["mapped"] & np.isfinite(result["dprime"])
    axes[0, 1].hist(result["dprime"][finite], bins=40, color=neutral)
    axes[0, 1].axvline(DP_THRESHOLD, color=leaf_color, linestyle="--")
    axes[0, 1].axvline(-DP_THRESHOLD, color=circle_color, linestyle="--")
    axes[0, 1].set(title="Signed per-neuron d′", xlabel="d′: leaf1 − circle1", ylabel="neurons")

    axes[1, 0].scatter(
        result["mean_circle"][finite], result["mean_leaf"][finite],
        s=8, color=neutral, alpha=0.4,
    )
    axes[1, 0].scatter(
        result["mean_circle"][result["leaf_selective"]],
        result["mean_leaf"][result["leaf_selective"]],
        s=14, color=leaf_color, label="d′ ≥ 0.3",
    )
    axes[1, 0].scatter(
        result["mean_circle"][result["circle_selective"]],
        result["mean_leaf"][result["circle_selective"]],
        s=14, color=circle_color, label="d′ ≤ −0.3",
    )
    axes[1, 0].set(
        title="Whole-session mean responses",
        xlabel="circle1 mean", ylabel="leaf1 mean",
    )
    axes[1, 0].legend(frameon=False)

    axes[1, 1].scatter(
        result["cortical_x"][result["mapped"]],
        result["cortical_y"][result["mapped"]],
        s=4, color=neutral, alpha=0.25,
    )
    axes[1, 1].scatter(
        result["cortical_x"][result["leaf_selective"]],
        result["cortical_y"][result["leaf_selective"]],
        s=14, color=leaf_color, label="leaf1",
    )
    axes[1, 1].scatter(
        result["cortical_x"][result["circle_selective"]],
        result["cortical_y"][result["circle_selective"]],
        s=14, color=circle_color, label="circle1",
    )
    axes[1, 1].set(
        title="Paper-coordinate locations",
        xlabel="cortical x = -xy_t[:, 1]",
        ylabel="cortical y = xy_t[:, 0]",
    )
    axes[1, 1].set_aspect("equal")
    axes[1, 1].legend(frameon=False)
    figure.suptitle(f"{RECORDING_ID} · released 1,000-neuron example")
    return figure


example_figure = plot_released_example(example_result)
example_figure


## What this output establishes

The output reports the released example's frame support, signed d′ values, and
paper-coordinate neuron locations. It does not estimate learning, reward effects,
or population prevalence because it contains one example recording and 1,000
released neurons. The paper's cohort-level result and sample sizes are in
[Figure 1](https://www.nature.com/articles/s41586-025-09180-y#Fig1) and the associated
[Results subsection](https://www.nature.com/articles/s41586-025-09180-y#Sec2).
